# 04 — Generate and Evaluate Answers

This notebook generates answers for all (question × strategy) pairs with `gpt-4o-mini`, then runs an LLM-as-judge evaluation (`gpt-4o-mini`, temperature 0) against **reference answers** and the **chunks each strategy actually retrieved**, and classifies any errors.

> Reference answers come from `src/reference_answers.py` (XBRL facts + gold filing sections, written by gpt-4o). Build and review them before running section 4.4.

**Pipeline:**
1. Load retrieval results from notebook 03
2. For each (question_id, strategy), call `generation.generate_answer()` with the retrieved chunks as context
3. Parse the structured JSON response (answer, citations, confidence, evidence_used)
4. Run `evaluation.evaluate_answer()` to score on 3 dimensions (0–2 each)
5. Run `evaluation.classify_error()` to assign an error type and stage
6. Save to `data/processed/evaluation_results.csv`

**Cost estimate:** ~120 generation calls + 120 evaluation calls ≈ ~$0.10–0.20 with gpt-4o-mini.

**Runtime:** ~15–25 minutes.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import pandas as pd

from src.config import PROCESSED_DIR
from src.generation import generate_answer, run_experiment
from src.evaluation import evaluate_answer, classify_error, run_full_evaluation
from src.error_taxonomy import ERROR_TYPES

questions_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'questions.csv'))
retrieval_df = pd.read_parquet(os.path.join(PROCESSED_DIR, 'retrieval_results.parquet'))

print(f"Questions: {len(questions_df)}")
print(f"Retrieval rows: {len(retrieval_df)}")
print(f"Strategies: {retrieval_df['strategy'].unique()}")

## 4.1 — Single answer demo

Test the generation pipeline on one question before running the full batch.

In [ ]:
demo_q = questions_df[questions_df['task_type'] == 'single_fact'].iloc[0]
question_id = demo_q['question_id']
question = demo_q['question']

demo_chunks = retrieval_df[
    (retrieval_df['question_id'] == question_id) &
    (retrieval_df['strategy'] == 'metadata_filtered')
].sort_values('rank').to_dict('records')

print(f"Question: {question}")
print(f"Retrieved {len(demo_chunks)} chunks")

result = generate_answer(question, demo_chunks)

print("\n--- Answer ---")
print(result['answer'])
print(f"\nConfidence: {result['confidence']}")
print(f"Evidence used: {result['evidence_used']}")
print(f"Citations: {json.dumps(result['citations'], indent=2)}")

## 4.2 — Run full generation experiment

In [ ]:
gen_results_path = os.path.join(PROCESSED_DIR, 'generation_results.csv')

if os.path.exists(gen_results_path):
    print(f"Loading existing generation results from {gen_results_path}")
    gen_df = pd.read_csv(gen_results_path)
else:
    print("Running generation experiment (this will call the OpenAI API)...")
    gen_df = run_experiment(questions_df, retrieval_df)
    gen_df.to_csv(gen_results_path, index=False)
    print(f"Saved to {gen_results_path}")

print(f"\nGeneration results shape: {gen_df.shape}")
print(f"Errors: {gen_df['error'].notna().sum()} / {len(gen_df)}")
gen_df.head(3)

## 4.3 — Confidence distribution

Check the model's self-reported confidence before running evaluation.

In [ ]:
import matplotlib.pyplot as plt

conf_pivot = gen_df.groupby(['strategy', 'confidence']).size().unstack(fill_value=0)
conf_pivot.plot(kind='bar', figsize=(10, 4), rot=0)
plt.title('Self-Reported Confidence by Strategy')
plt.xlabel('Strategy')
plt.ylabel('Count')
plt.legend(title='Confidence')
plt.tight_layout()
plt.show()

## 4.4 — Run full evaluation (LLM-as-judge)

In [ ]:
eval_results_path = os.path.join(PROCESSED_DIR, 'evaluation_results.csv')

if os.path.exists(eval_results_path):
    print(f"Loading existing evaluation results from {eval_results_path}")
    eval_df = pd.read_csv(eval_results_path)
else:
    print("Running LLM-as-judge evaluation...")
    from src.reference_answers import build_reference_answers, attach_reference_answers
    build_reference_answers()                      # no-op if reference_answers.csv already exists
    questions_df = attach_reference_answers()
    eval_df = run_full_evaluation(gen_df, questions_df, retrieval_df)

print(f"\nEvaluation results shape: {eval_df.shape}")
print(f"\nMean scores by strategy:")
print(eval_df.groupby('strategy')[['answer_correctness', 'citation_faithfulness', 'evidence_coverage']].mean().round(2))

## 4.5 — Error distribution preview

In [ ]:
print("Error type distribution:")
print(eval_df['error_type'].value_counts())

print("\nError stage distribution:")
print(eval_df['error_stage'].value_counts())

print("\nError stage by strategy:")
print(eval_df.groupby(['strategy', 'error_stage']).size().unstack(fill_value=0))